# Dafne Thigh Segmentation — Sheffield Dataset (Lambda)

Runs the **Dafne Thigh model** 2D slice-by-slice on each of the 69 Sheffield augmented
DICOM volumes.  Input is the single-channel greyscale image (used as a water-image proxy).

Data: `~/sheffeld/20440164/Aug_N.dcm`  (N = 1 … 69, multi-frame greyscale DICOM)
Output: `~/dafne_sheffield_segs/Aug_N/Aug_N_dafne_thigh.npz`

## 1 — Upload to Lambda
```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/sheffeld \
  ubuntu@<YOUR-LAMBDA-IP>:~/

rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  "/tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/dafne/model/" \
  ubuntu@<YOUR-LAMBDA-IP>:~/dafne_model/
```

## 2 — Download results
```bash
rsync -avz --mkpath -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  ubuntu@<YOUR-LAMBDA-IP>:~/dafne_sheffield_segs/ \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/dafne/sheffield_segs/
```
**Terminate the instance when done.**

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'dafne-dl', 'SimpleITK', 'pydicom'])
print('Dependencies installed.')

In [ ]:
import glob, os, re
import numpy as np
import SimpleITK as sitk
import pydicom
from dafne_dl import DynamicDLModel

IMG_DIR    = os.path.expanduser('~/sheffeld/20440164')
OUTPUT_DIR = os.path.expanduser('~/dafne_sheffield_segs')

_model_candidates = sorted(glob.glob(os.path.expanduser('~/dafne_model/*.model')))
if not _model_candidates:
    raise FileNotFoundError('No .model file found in ~/dafne_model/ — upload it first')
MODEL_PATH = _model_candidates[0]

dcm_files = sorted(
    [f for f in glob.glob(os.path.join(IMG_DIR, 'Aug_*.dcm'))
     if '_segmentations' not in f],
    key=lambda p: int(re.search(r'Aug_(\d+)\.dcm', p).group(1)),
)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Model : {MODEL_PATH}')
print(f'Found : {len(dcm_files)} DICOM volumes')

In [ ]:
model = DynamicDLModel.Load(open(MODEL_PATH, 'rb'))
print('Model loaded.')

In [ ]:
def read_dicom_volume(dcm_path):
    """Return (array float32 D×H×W, in-plane spacing [sx, sy])."""
    ds  = pydicom.dcmread(dcm_path)
    arr = ds.pixel_array.astype(np.float32)
    ps  = getattr(ds, 'PixelSpacing', [1.0, 1.0])
    return arr, [float(ps[0]), float(ps[1])]

for dcm_path in dcm_files:
    idx = re.search(r'Aug_(\d+)\.dcm', dcm_path).group(1)
    out_subdir = os.path.join(OUTPUT_DIR, f'Aug_{idx}')
    out_path   = os.path.join(out_subdir, f'Aug_{idx}_dafne_thigh.npz')

    if os.path.exists(out_path):
        print(f'Skipping (done): Aug_{idx}')
        continue

    print(f'\nProcessing: Aug_{idx}')
    os.makedirs(out_subdir, exist_ok=True)

    img_array, resolution = read_dicom_volume(dcm_path)
    D, H, W = img_array.shape
    print(f'  Shape: {img_array.shape}  Resolution: {resolution}')

    all_masks = {}
    for sl in range(D):
        out = model({
            'image':            img_array[sl],
            'resolution':       resolution,
            'split_laterality': True,
            'classification':   'Thigh',
        })
        for name, mask in out.items():
            if name not in all_masks:
                all_masks[name] = np.zeros((D, H, W), dtype=np.uint8)
            all_masks[name][sl] = np.asarray(mask, dtype=np.uint8)
        if (sl + 1) % 50 == 0 or sl == D - 1:
            print(f'  slice {sl+1}/{D}')

    np.savez_compressed(out_path, **all_masks)
    print(f'  Saved → {out_path}  muscles: {list(all_masks.keys())}')

print('\nAll done.')

In [ ]:
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*', '*.npz')))
print(f'Output files: {len(results)} / {len(dcm_files)}')
if results:
    s = np.load(results[0])
    print(f'Sample: {results[0]}')
    for k in sorted(s.files):
        print(f'  {k}: {s[k].shape}  voxels={int(s[k].sum()):,}')